# G.I KO KO AI VIDEO STUDIO — Colab Cloud Renderer
Upload an MP4, select an output aspect ratio, and render with FFmpeg. A free T4 GPU is not guaranteed; CPU fallback is automatic.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu-core

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Please upload an MP4 video first.')
input_path = next(iter(uploaded.keys()))
print('Input:', input_path)

In [ ]:
import subprocess, shutil
gpu = False
if shutil.which('nvidia-smi'):
    try:
        subprocess.run(['nvidia-smi'], check=True, stdout=subprocess.DEVNULL)
        encoders = subprocess.run(['ffmpeg','-hide_banner','-encoders'], capture_output=True, text=True, check=True).stdout
        gpu = 'h264_nvenc' in encoders
    except Exception:
        gpu = False
print('NVIDIA NVENC encoder available:' if gpu else 'GPU encoder unavailable; using CPU:', gpu)

In [ ]:
# Change these settings before rendering
ASPECT_RATIO = '9:16'  # choose: '9:16', '16:9', '1:1'
WATERMARK = 'Powered by I KO KO'
OUTPUT_PATH = 'GIKOKO_RENDERED.mp4'
sizes = {'9:16': '1080:1920', '16:9': '1920:1080', '1:1': '1080:1080'}
if ASPECT_RATIO not in sizes:
    raise ValueError('Invalid ASPECT_RATIO')
size = sizes[ASPECT_RATIO]
Path('watermark.txt').write_text(WATERMARK, encoding='utf-8')
encoder = 'h264_nvenc' if gpu else 'libx264'
cmd = ['ffmpeg','-y','-i',input_path,'-vf',f"scale={size}:force_original_aspect_ratio=decrease,pad={size}:(ow-iw)/2:(oh-ih)/2,drawtext=fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf:textfile=watermark.txt:fontcolor=white:fontsize=32:box=1:boxcolor=black@0.45:boxborderw=12:x=w-tw-30:y=h-th-30"]
if encoder == 'h264_nvenc':
    cmd += ['-c:v','h264_nvenc','-preset','p4','-cq','23']
else:
    cmd += ['-c:v','libx264','-preset','veryfast','-crf','23']
cmd += ['-c:a','aac','-b:a','192k','-movflags','+faststart',OUTPUT_PATH]
print('Encoder:', encoder)
subprocess.run(cmd, check=True)
print('Render complete:', OUTPUT_PATH)

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)